# scarlatti-doodle project!

## Zip 파일 메모리에 로딩
#### this code block was written using AI

In [6]:
import io
import os
import zipfile
import tempfile
import partitura as pt
from music21 import midi, environment

# music21 임시 디렉토리 억까 방지 설정
try:
    environment.Environment()['directoryScratch'] = '/tmp'
except:
    pass

def mount_scarlatti():
    """scarlatti.zip을 가상 메모리에 마운트하고, 파일 목록을 반환합니다."""
    ZIP_FILE_PATH = "original_midi.zip"
    
    if not os.path.exists(ZIP_FILE_PATH):
        raise FileNotFoundError(f"⚠️ '{ZIP_FILE_PATH}' 파일이 없습니다. 왼쪽 탐색기에 업로드해 주세요!")
        
    print("🛸 가상 메모리(RAM)에 스칼라티 555개 소나타 마운트 중...")
    
    with open(ZIP_FILE_PATH, "rb") as f:
        zip_buffer = io.BytesIO(f.read())
        
    archive = zipfile.ZipFile(zip_buffer)
    midi_files = [f for f in archive.namelist() if f.lower().endswith(('.mid', '.midi'))]
    print(f"📦 마운트 완료! 총 {len(midi_files)}개의 가상 미디 파일 준비 완료.")
    
    return archive, midi_files

def load_midi_from_virtual_folder(archive, file_path, target_type="partitura"):
    """
    [에러 수정 완료] 가상 폴더에서 데이터를 읽어 지정한 라이브러리 객체로 변환합니다.
    BytesIO 타입 제한 억까를 우회하기 위해 OS 임시 파일 핸들러를 안전하게 사용합니다.
    """
    # 1. 압축 파일 내부에서 순수 바이트 데이터 추출
    midi_raw_bytes = archive.read(file_path)
    
    # 2. 파르티투라 객체로 변환할 때
    if target_type == "partitura":
        # 💡 [핵심 우회] BytesIO를 거부하므로, RAM처럼 작동하는 임시 파일을 시스템에 잠깐 썼다 지웁니다.
        # 디스크에 영구 저장되지 않고 함수가 끝나면 메모리에서 자동 소멸해서 속도가 엄청 빠릅니다!
        with tempfile.NamedTemporaryFile(delete=False, suffix=".mid") as tmp_file:
            tmp_file.write(midi_raw_bytes)
            tmp_file_path = tmp_file.name
        
        try:
            # 안전하게 문자열 경로(PathLike)로 인식시켜서 에러 타파!
            performance = pt.load_performance_midi(tmp_file_path)[0]
        finally:
            # 처리가 끝나면 임시 파일 흔적 지우기
            if os.path.exists(tmp_file_path):
                os.remove(tmp_file_path)
                
        return performance
        
    # 3. 뮤직21 객체로 변환할 때 (기존과 동일, 잘 작동함)
    elif target_type == "music21":
        mf = midi.MidiFile()
        mf.readstr(midi_raw_bytes)
        m21_score = midi.translate.midiFileToStream(mf)
        return m21_score
        
    else:
        raise ValueError("⚠️ target_type은 'partitura' 또는 'music21'만 가능합니다!")

# 🔥 [실행] 가상 폴더 연결하기
scarlatti_folder, file_list = mount_scarlatti()
print("앞으로 scarlatti_folder, file_list에 접근해서 사용!")

🛸 가상 메모리(RAM)에 스칼라티 555개 소나타 마운트 중...
📦 마운트 완료! 총 555개의 가상 미디 파일 준비 완료.
앞으로 scarlatti_folder, file_list에 접근해서 사용!


## 전처리 함수들 준비

In [7]:
from music21 import *
import partitura as pt
import random
import numpy as np



# 흐름
# 미디파일 -> 퀀타이즈 -> 모든조로 전조 -> 렌덤으로 마디 분리 -> 오른손 왼손 분리 -> 오른손 데이터는 마스킹 -> 마스킹한 오른손 데이터는 인풋으로, 온전한 오른손과 왼손 데이터는 아웃풋으로 넘파이 어레이로 전환해서 ai모델 학습용 리스트에 저장

def Score_Quantize(partituraPerformance: pt.performance.PerformedPart):
    '''25ms 단위로 음들 퀀타이즈해서 리턴하는 함수'''
    note_array = partituraPerformance.note_array().copy()
    for i in range(len(note_array)):
        note_array[i]['onset_sec'] = round(note_array[i]['onset_sec'] * 1000 / 25) * 25 / 1000
        note_array[i]['duration_sec'] = max(0.025, round(note_array[i]['duration_sec'] * 1000 / 25) * 25 / 1000)

    return pt.performance.PerformedPart.from_note_array(note_array)



def Score_TrebleBassSeparation_Partitura(partituraPerformance: pt.performance.PerformedPart, splitPoint: int = 60):
    '''높은음자리표 낮은음자리표 분리해서 리턴하는 함수'''
    performance = partituraPerformance
    note_array = performance.note_array()
    
    split_pitch = splitPoint
    
    treble_mask = note_array['pitch'] >= split_pitch
    bass_mask = note_array['pitch'] < split_pitch
    
    treble_note_array = note_array[treble_mask] # 이거는 단순 마스크! 프린트하면 true와 false 로 이루어진 어레이가 나옴!
    bass_note_array = note_array[bass_mask]
    
    # NumPy 배열을 partitura가 저장할 수 있는 PerformedPart 객체 상자에 다시 담아줍니다.
    treble_part = pt.performance.PerformedPart.from_note_array(treble_note_array)
    bass_part = pt.performance.PerformedPart.from_note_array(bass_note_array)
    
    return treble_part, bass_part


def Score_TransposeToAllKeys(partituraPerformance : pt.performance.PerformedPart, music21Score : stream.Score):
    '''스코어 파일을 모든 조로 전조해서 리턴'''
    transposedPerformances = []
    num = PredictedKeyToNum(music21Score=music21Score)

    note_array = partituraPerformance.note_array().copy()
    note_array["pitch"] -= num

    for _ in range(0, 12):
        transposedPerformances.append(pt.performance.PerformedPart.from_note_array(note_array.copy())) # 제미나이가 넣으라고 해서 넣었는거인데 추후에 확인해보기
        note_array["pitch"] += 1

    return transposedPerformances

def PredictedKeyToNum(music21Score : stream.Score):
    KEY_TO_NUM = {
    'C': 0, 'C#': 1, 'D': 2, 'D#': 3, 'E': 4, 'F': 5,
    'F#': 6, 'G': 7, 'G#': 8, 'A': 9, 'A#': 10, 'B': 11,
    'Db': 1, 'Eb': 3, 'Gb': 6, 'Ab': 8, 'Bb': 10
    }

    predicted_key = str(music21Score.analyze('key').tonic.name)
    num = KEY_TO_NUM[predicted_key.upper().replace('-', 'b')]

    return num


def Score_SliceByMeasures(partituraPerformance : pt.performance.PerformedPart):
    '''스코어 파일을 30초 길이만큼 나눠서 리턴'''
    # 미디를 30초 단위로 나눔, 0에서 2 사이의 마디만큼 겹침, 남은 마디가 부족하면 겹쳐서라도 16마디 맟춤
    slicedPerformances = []
    note_array = partituraPerformance.note_array().copy()
    maxSec = (note_array['onset_sec'] + note_array['duration_sec']).max()
    
    endSec = 30.0

    while(endSec <= maxSec):

        mask = ((endSec - 30.0) <= note_array['onset_sec']) & (note_array['onset_sec'] <= endSec)
        slicedPerformance = note_array[mask].copy() # 주어진 범위 안에 들어가는 음들만 정리

        slicedPerformanceMask = (slicedPerformance['onset_sec'] + slicedPerformance['duration_sec']) > endSec # 범위 안 음들 중 끝 범위 넘어가는 것들 찾아내기
        slicedPerformance['duration_sec'][slicedPerformanceMask] = endSec - slicedPerformance['onset_sec'][slicedPerformanceMask] # 끝 범위 넘어가는 음들은 길이 조정
        slicedPerformance['onset_sec'] -= (endSec - 30.0) # 시작점 0으로 맞춤

        slicedPerformances.append(pt.performance.PerformedPart.from_note_array(slicedPerformance)) # 잘라낸 음들을 partitura 객체로 변환해서 리스트에 추가

        if(maxSec == endSec):
            break
        elif(maxSec - endSec < 30.0): 
            endSec = maxSec
        else:
            endSec = endSec + 30.0 - random.randint(0, 15)
    
    return slicedPerformances


def Score_MaskNotes(partituraTrebblePerformance : pt.performance.PerformedPart):
    '''멜로디 데이터에서 일부 데이터들 마스킹해서 리턴'''
    note_array = partituraTrebblePerformance.note_array().copy()
    
    survived_note_array = []

    randNum = random.randint(1, 20)

    if(randNum <= 10):
        threshold1, threshold2, threshold3 = 4, 8, 9
    elif(randNum <= 14):
        threshold1, threshold2, threshold3 = 3, 6, 8
    elif(randNum <= 17):
        threshold1, threshold2, threshold3 = 2, 8, 9
    elif(randNum <= 19):
        threshold1, threshold2, threshold3 = 5, 6, 7
    else:
        threshold1, threshold2, threshold3 = 7, 8, 9

    for i in range(0, len(note_array)):
        randNum = random.randint(1, 10)
        if(randNum <= threshold1): # 그대로두기
            survived_note_array.append(note_array[i])
        elif(randNum <= threshold2): # 없애기
            continue
        elif(randNum <= threshold3): # 피치 변형하기
            note_array[i]['pitch'] = min(127, max(0, int(note_array[i]['pitch'] + random.randint(-12, 12))))
            survived_note_array.append(note_array[i])
        else: # 늘이거나 줄이기
            note_array[i]['onset_sec'] = max(round(random.uniform(-0.2, 0.2),6) + note_array[i]['onset_sec'], 0.000001)
            note_array[i]['duration_sec'] = max(round(random.uniform(-0.2, 0.2),6) + note_array[i]['duration_sec'], 0.05)
            survived_note_array.append(note_array[i])

    # onset_sec(x[0])을 기준으로 시간순 정렬!
    survived_note_array.sort(key=lambda x: x['onset_sec'])

    if(len(survived_note_array) == 0):
        return Score_MaskNotes(partituraTrebblePerformance=partituraTrebblePerformance)

    return pt.performance.PerformedPart.from_note_array(np.array(survived_note_array, dtype=note_array.dtype))

def ScoreToDataset(partituraPerformance : pt.performance.PerformedPart):
    '''스코어 파일을 ai 학습용 데이터셋으로 변환해서 리턴'''
    array = np.zeros((128, 40 * 30))
    note_array = partituraPerformance.note_array().copy()

    for i in range(len(note_array)):
        pitch = note_array[i]['pitch']
        startPoint = int(round(note_array[i]['onset_sec'] * 1000 / 25))
        endPoint = int(round((note_array[i]['onset_sec'] + note_array[i]['duration_sec']) * 1000 / 25))

        if(startPoint >= 1200):
            continue
        endPoint = min(endPoint, 1200)

        array[pitch, startPoint] = 2 # 음이 시작하는 부분은 2로 설정해서 표시
        startPoint += 1

        if(startPoint < endPoint):
            array[pitch, startPoint:endPoint] = 1

    return array

def DatasetToScore(dataset : np.ndarray):
    '''ai가 출력한 데이터셋을 스코어 파일로 변환해서 리턴'''
    note_list = []

    indices = np.where(dataset == 2)
    coordinates = list(zip(indices[0], indices[1])) # 음이 시작하는 위기들 모두 찾기
    for coordinate in coordinates:
        coordinatePitch, coordinateStartPoint = coordinate
        endPoint = 1
        while(coordinateStartPoint + endPoint < 1200 and dataset[coordinatePitch][coordinateStartPoint + endPoint] == 1):
            endPoint += 1
        
        # 반복문 돌면서 복원한 값 집어넣기
        note_list.append((
            coordinateStartPoint / 40.0,
            endPoint / 40.0,
            int(coordinatePitch),
            127
        ))
    
    fields = [('onset_sec', 'f8'), ('duration_sec', 'f8'), ('pitch', 'i4'), ('velocity', 'i4')]
    note_array = np.array(note_list, dtype=fields)

    return pt.performance.PerformedPart.from_note_array(note_array)



    


## GAN 모델 학습시키기

### 1. 학습용 데이터셋들 준비

In [ ]:
import shutil

num = 1
save_dir = "processed_data"

if os.path.exists(save_dir):
    shutil.rmtree(save_dir)

os.makedirs(os.path.join(save_dir, "X"), exist_ok=True)
os.makedirs(os.path.join(save_dir, "Y"), exist_ok=True)

for filepath in file_list: # TODO 파일들 너무 많이 만들어져서 일단 한개 미디만 불러오게 해놓음 나중에 학습시킬때는 지우기! (지움)
    performance = load_midi_from_virtual_folder(scarlatti_folder, filepath)
    performance = Score_Quantize(partituraPerformance=performance) # 퀀타이즈
    score = load_midi_from_virtual_folder(scarlatti_folder, filepath,target_type="music21")
    transposed_list = Score_TransposeToAllKeys(partituraPerformance=performance,music21Score=score) # 모든조로 전조
    for i in range(len(transposed_list)):
        slicedPerformances = Score_SliceByMeasures(partituraPerformance=transposed_list[i]) # 30초단위로 나눔
        for slicedPerformance in slicedPerformances: 
            keyNum = PredictedKeyToNum(score)
            trebble, bass = Score_TrebleBassSeparation_Partitura(partituraPerformance=slicedPerformance, splitPoint = 60 - keyNum + i) # 오른손왼손 분리
            if trebble is None or len(trebble.note_array()) == 0:
                continue
            trebble = Score_MaskNotes(partituraTrebblePerformance=trebble)

            x = ScoreToDataset(trebble)
            y = ScoreToDataset(slicedPerformance)
            x = x.astype(np.uint8)
            y = y.astype(np.uint8)

            x_path =  os.path.join(save_dir, "X", f"sample_{num}.npz")
            y_path =  os.path.join(save_dir, "Y", f"sample_{num}.npz")
            np.savez_compressed(x_path, data=x)
            np.savez_compressed(y_path, data=y)

            num += 1


    

Matplotlib is building the font cache; this may take a moment.


### 2. 데이터 로드

In [9]:
import os
import glob
import numpy as np
import torch
from torch.utils.data import Dataset

class MelodyDataset(Dataset):
    def __init__(self, data_dir="processed_data"):
        # X 폴더와 Y 폴더 내의 npz(또는 npy) 파일 경로를 이름 순으로 정렬해서 수집
        self.x_files = sorted(glob.glob(os.path.join(data_dir, "X", "*.npz")))
        self.y_files = sorted(glob.glob(os.path.join(data_dir, "Y", "*.npz")))

        # X와 Y의 파일 개수가 일치하는지 체크
        assert len(self.x_files) == len(self.y_files), "X와 Y 폴더의 파일 개수가 안맞음"

    def __len__(self):
        return len(self.x_files)

    def __getitem__(self, idx):
        # x_files와 y_files에서 각각 파일 불러오기
        x_path = self.x_files[idx]
        y_path = self.y_files[idx]

        # np.load 처리
        x_data = np.load(x_path)
        y_data = np.load(y_path)

        # 만약 npz 형식이라 키값 추출이 필요한 경우를 대비 (npy 파일이면 바로 배열로 읽힘)
        x = x_data['data'] if isinstance(x_data, np.lib.npyio.NpzFile) else x_data
        y = y_data['data'] if isinstance(y_data, np.lib.npyio.NpzFile) else y_data

        # PyTorch UNet 입력 포맷에 맞게 형변환
        # X: (1, Pitch_Height, Time_Steps) -> float32
        x_tensor = torch.tensor(x, dtype=torch.float32).unsqueeze(0)
        
        # Y: (Pitch_Height, Time_Steps) -> long (값: 0, 1, 2)
        y_tensor = torch.tensor(y, dtype=torch.long)

        return x_tensor, y_tensor


### 3. 모델 학습

In [10]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as optim

# 0. 맥북 장치 설정 (Apple Silicon GPU 가속)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu") # metal performance shaders 사용할 수 있으면 사용하고 못하면 걍 cpu 사용
print(f"⚙️ 사용 중인 디바이스: {device}")

# 1. UNet 블록 및 모델 정의
class DoubleConv(nn.Module): # doubleconv를 따로 클래스로 만들어 두는 이유는 이게 모델에서 여러번 쓰이기 때문!!
    def __init__(self, in_channels, out_channels):  # __init__은 클래스로부터 객체를 만들 때 자동으로 실행되는 초기화 함수(생성자)!!! 첫번째 인자는 지금 만들어지는 함수이고 나머지 인자들로 객체에 넣을 데이터 지정!
        super().__init__() # 부모 클래스의 객체를 반환 (상위 클래스의 초기화 함수를 자식 클래스에서 그대로 쓸 수 있게 해줌) 자식 클래스에서 부모 클래스의 초기화 함수를 그대로 실행
        self.conv = nn.Sequential( # nn.Sequential은 여러 신경망 층(layer)을 순서대로 감싸는 순차 컨테이너 모듈, 처리 과정을 뭉쳐서 변수에 담을 수 있게 만듦 레이어가 가진 가중치를 계속 저장하기 위에 변수에 저장
            # 변수 선언 시 앞에 self. 붙이는 이유는 안붙이면 init 함수가 끝나면 바로 사라지기 때문
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1), # 여기서 쓰인 인자는 인풋, 아웃풋, 커널 사이즈(필터의 크기, 여기서는 3이므로 3 x 3 필터),  입력 이미지 테두리에 추가할 0 등의 픽셀 수인 padding (출력 크기 감소 방지, 테두리 정보 보호, 모델을 더 깊게 만들기 위해)
            nn.BatchNorm2d(out_channels), # 데이터가 너무 크거나 작아지지 않도록 평균을 1 분산을 0으로 해서 고르게 펴줌
            nn.ReLU(inplace=True), # 활성화함수 (inplace=True 는 연산 결과를 새로운 메모리에 저장하지 않고, 입력받은 원본 텐서의 데이터를 직접 덮어쓰기하겠다는 뜻)
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x): # 파이토치 모듈에서 순전파를 자동으로 실행하게 되어 있어서 순전파 함수를 만들어주어야 함
        return self.conv(x)

class MelodyUNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=3): # 여기서 num_classes는 음 시작은 2 지속은 1 없는거는 0 구분용 그래서 채널이 3개가 나오는거
        super().__init__()

        self.inc = DoubleConv(in_channels, 32) # 위에서 만든 변수를 inc(input convolution, 모델에서 처음으로 거치는 블록) 변수에 할당
        self.pool1 = nn.MaxPool2d(2) # 커널 사이즈를 2로 지정해서 풀링층을 통과

        self.down1 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(64, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2) # 일반 컨볼루션의 반대로 진행되는 전치 합성곱(Transposed Convolution)을 진행, 인자는 인풋 아웃풋 커널 사이즈와 필터가 이동하는 간격인 stride(여기서는 stride 가 2기 때문에 이미지가 2배 커짐)
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(64, 32)

        self.outc = nn.Conv2d(32, num_classes, kernel_size=1) # outc 는 out convolution이라는 뜻!! 커널 크기를 1로 해서 특징만 요약!!

    def forward(self, x):
        # ********인코더********
        x1 = self.inc(x) # input convolution 변수에 인풋값(x) 집어넣어서 할당
        x2 = self.down1(self.pool1(x1)) # 인코더(풀링층1(x1)) 구조
        
        x_bottleneck = self.bottleneck(self.pool2(x2)) # doubleconv(풀링층2(x2)) 구조

        #********디코더********
        x = self.up2(x_bottleneck) # transposed convolution에 x_bottleneck 넣어서 실행
        x = torch.cat([x, x2], dim=1) # 두 텐서를 1번 차원(dim) 즉 열 방향으로 이어붙임 (cat은 concatenate의 약자)
        x = self.dec2(x) # x를 doubleconv에 대입

        x = self.up1(x)
        x = torch.cat([x, x1], dim=1)
        x = self.dec1(x)

        logits = self.outc(x) # logits이란 모델의 마지막 층에서 활성화 함수(시그모이드나 소프트맥스 등)를 거치기 전 상태인 원시 점수(raw score)
        return logits

# 2. 데이터 불러오기 (num_workers=0 적용)
dataset = MelodyDataset(data_dir="processed_data") #  processed_data 를 경로로 만들어둔 클래스 인스턴스화
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=0) # 파이토치 기본 클래스 Dataloader를 인스턴스화 (인자는 각각 dataset 객체, 한 번에 넣을 데이터 수 크기인 batch_size, 순서를 섞을 지 여부인 shuffle, 데이터를 로딩할 때 사용할 서브 프로세스(cpu 코어)의 개수인 num_workers)

# 3. 첫 번째 배치 꺼내기
x_batch, y_batch = next(iter(dataloader)) # dataloader 는 반복문으로 순환하는 객체인데 이걸 하나씩 꺼내는 iterater로 변환해서 next()로 첫 번째 배치를 가져옴

# TODO 여기 아래로 주석 달기
model = MelodyUNet(in_channels=1, num_classes=3).to(device) # 모델에 MelodyUNet 클래스 할당하고 인풋 채널 수, 분류할 클래스 수 지정하고 처리할 장치로 전달

criterion = nn.CrossEntropyLoss() # 모델이 예측한 값과 실제 값의 오차를 계산하는 함수 불러옴

optimizer = optim.Adam(model.parameters(), lr=0.001) # 옵티마이저에 adam 사용 (model.parameters()는 최적화 대상이 될 모델 안의 모든 가중치와 편향, lr은 learning rate로 한번에 가중치를 얼마나 크게 바꿀지 정함)

num_epochs = 30  # 전체 데이터를 30번 반복해서 학습

print("\n학습 시작")

for epoch in range(num_epochs):
    model.train()  # 모델을 학습 모드로 설정
    running_loss = 0. # 누적 오차 값을 0으로 초기화

    for x_batch, y_batch in dataloader:
        x_batch = x_batch.to(device).float()  # 입력은 float
        y_batch = y_batch.to(device).long()  # 정답(0,1,2)은 정수(long)

        optimizer.zero_grad() # 이전 스텝의 기울기 초기화

        logits = model(x_batch) # 모델에 입력 데이터 넣어서 logits(활성화함수 거치기 전의 날것의 점수) 에 할당

        loss = criterion(logits, y_batch) # 손실 함수로 예측한 값과 실제 값의 오차 구함 (위에서 불러온 crossentropyloss 손실함수 씀)

        loss.backward() # 역전파 수행

        optimizer.step() # 옵티마이저가 가중치 값 수정

        running_loss += loss.item() # 누적 오차에 이번 배치의 손실값 더함 (.item()은 텐서 안에 든 순수 숫자만 꺼내는 함수)

    # 1 Epoch 마다 평균 Loss 출력
    epoch_loss = running_loss / len(dataloader)
    print(f"Epoch [{epoch+1:02d}/{num_epochs}] ── Loss: {epoch_loss:.4f}") # epoch 최소 2자리 정수로 맞추고, loss는 소수점 4번째 자리까지만 반올림해서 나타냄

print("\n학습 완료")

# 완성된 모델 저장
torch.save(model.state_dict(), "melody_unet_best.pth")
print("'melody_unet_best.pth'저장됨")

⚙️ 사용 중인 디바이스: mps

학습 시작
Epoch [01/30] ── Loss: 0.8923
Epoch [02/30] ── Loss: 0.5655
Epoch [03/30] ── Loss: 0.4104
Epoch [04/30] ── Loss: 0.3063
Epoch [05/30] ── Loss: 0.2348
Epoch [06/30] ── Loss: 0.1878
Epoch [07/30] ── Loss: 0.1567
Epoch [08/30] ── Loss: 0.1354
Epoch [09/30] ── Loss: 0.1204
Epoch [10/30] ── Loss: 0.1096
Epoch [11/30] ── Loss: 0.1015
Epoch [12/30] ── Loss: 0.0956
Epoch [13/30] ── Loss: 0.0907
Epoch [14/30] ── Loss: 0.0874
Epoch [15/30] ── Loss: 0.0845
Epoch [16/30] ── Loss: 0.0820
Epoch [17/30] ── Loss: 0.0802
Epoch [18/30] ── Loss: 0.0784
Epoch [19/30] ── Loss: 0.0771
Epoch [20/30] ── Loss: 0.0762
Epoch [21/30] ── Loss: 0.0752
Epoch [22/30] ── Loss: 0.0745
Epoch [23/30] ── Loss: 0.0734
Epoch [24/30] ── Loss: 0.0730
Epoch [25/30] ── Loss: 0.0721
Epoch [26/30] ── Loss: 0.0715
Epoch [27/30] ── Loss: 0.0707
Epoch [28/30] ── Loss: 0.0704
Epoch [29/30] ── Loss: 0.0697
Epoch [30/30] ── Loss: 0.0692

학습 완료
'melody_unet_best.pth'저장됨


### 4. 모델 실행

In [ ]:
import partitura as pt
import numpy as np
import torch

test_performance = pt.load_performance_midi("In My Life.MID") # 미디 파일 불러와서 partitura performance로 변환

input_data = ScoreToDataset(test_performance) # 넘파이 어레이로 변환
input_tensor = torch.from_numpy(input_data).float() # 4d 텐서로 변환

# 배치(1)와 채널(1) 차원 추가 (4차원으로 만듦)
if input_tensor.ndim == 2:  # 2차원일때([128, 1200] 인 경우)
    input_tensor = input_tensor.unsqueeze(0).unsqueeze(0)  # [1, 1, 128, 1200]
elif input_tensor.ndim == 3:  # 3차원일때([1, 128, 1200] 인 경우)
    input_tensor = input_tensor.unsqueeze(0)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu") #디바이스 설정
model = MelodyUNet(in_channels=1, num_classes=3).to(device) # 만들어둔 모델 불러옴

checkpoint = torch.load("melody_unet_checkpoint.pth", map_location=device) # 딥러닝 모델 파일 불러옴 (map_location은 모델이 저장되었을때의 장치와 지금 실행하는 컴퓨터의 장치가 달라도 에러 없이 지정한 장치로 매핑하여 불러올 수 있게 하는거)
model.load_state_dict(checkpoint) # 미리 저장해둔 가중치와 평향을 모델 객체에 덮어씌워 모델의 상태를 복원

model.eval() # 평가 모드

input_tensor = input_tensor.to(device) # 입력 텐서도 모델과 같은 디바이스로 이동

with torch.no_grad():
    logits = model(input_tensor)
    probs = torch.sigmoid(logits)

probs_matrix = probs.squeeze(0).cpu().numpy()

# 음 시작, 음 지속 기준을 다르게 할 수 있음
ONSET_THRESHOLD = 0.50   # 음 시작
SUSTAIN_THRESHOLD = 0.55 # 음 지속

onset_mask = probs_matrix[1] > ONSET_THRESHOLD
sustain_mask = probs_matrix[2] > SUSTAIN_THRESHOLD # 불리언 마스킹

result_matrix = np.zeros_like(probs_matrix[0], dtype=np.int64) # probs_matrix[0] 모양으로 영행렬 만듦
result_matrix[sustain_mask] = 2
result_matrix[onset_mask] = 1 # 만들어두었던 불리언 마스크로 숫자 지정

result_performance = DatasetToScore(result_matrix)
pt.save_performance_midi(result_performance, "output_performance.mid") # 미디파일로 저장